# Part I: Dataset agreement and original validation, 2013–2025

This is the sole active `00_dataset` notebook. It combines the existing corpus-agreement workflow with the locked development/held-out classifier validation in Part II. The two exact source notebooks are preserved under `src/data_analysis/_archived/`.

The publication window is **1 January 2013–31 December 2025 inclusive**. Part I recalculates current corpus diagnostics from dated inputs; Part II defaults to the frozen model outputs supplied with the project and regenerates every held-out figure without requiring a GPU, network access, or Hugging Face token. Set `UKB_RERUN_HELDOUT_MODELS=1` only for an explicit full model rerun.

The original validation panels retained in Part I are historical diagnostics, not the final classifier-performance analysis. Primary classifier claims come only from Part II: prompt selection uses the development split, while precision, recall, F1, confusion matrices and pairwise agreement use the untouched held-out split.

| Figure family | Content | Export location/prefix |
|---|---|---|
| Original manuscript layout, 2×2 | Annual model positives; consensus/rest counts; keyword profiles; semantic map and silhouette | `00_06_figure_consensus_validation` |
| Original six-prompt agreement, 2×3 | Classifier agreement for each original prompt strategy | `00_04_figure_validation_agreement` |
| Original validation performance, 2×2 | Accuracy, precision, recall and F1 | `00_03_figure_validation_performance` |
| Candidate diagnostics | Agreement, text profiles, semantic diagnostics and consensus categories | `00_01`, `00_02`, `00_05`, `00_07` |
| Locked held-out agreement | Six individual prompt heatmaps and one combined 2×3 figure | `pairwise_agreement_*` |
| Locked held-out performance | Precision/recall/F1 heatmaps and selected-prompt bars | `heldout_*performance*` |
| Locked held-out confusion matrices | One 6×6 overview and six prompt-specific figures | `heldout_confusion_matrices_*` |

Part I figures use the shared project style and are written below `output/figures/data_analysis/00_dataset/`. Part II uses the same project palette and retains its original reproducibility bundle below `output/validation/ukb_prompt_validation_heldout_v3/`.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()


## 1. Inputs and run options

Part A automatically reads `data/analysis/dataset/matched_ukb_full_final_2013_2025_three_model_labels.csv`. `UKB_COMBINED_LABELS_CSV` can override it with another full candidate-level CSV. Showcase+ and TRUE-only exports cannot replace the candidate pool.

**Semantic analysis is enabled**, as in the original notebook. Matching coordinates and metrics are reused first. Otherwise a balanced sample of up to 3,000 papers per group is embedded using MiniLM; this never reruns Qwen, Llama or Mistral. `SEMANTIC_LOCAL_ONLY=True` uses cached model weights without downloads. If the encoder cannot load, the original explicitly labelled TF-IDF/SVD fallback remains available. Set `RUN_SEMANTIC_ANALYSIS=False` to reuse a saved semantic result without computing a new one.

Part B needs the six original `predictions_p*.csv` validation files under `VALIDATION_OUTPUT_DIR`. They are a different dataset from the three-model candidate labels. Existing predictions are reused. New validation inference remains disabled; enabling it requires the positive/negative labelled files and explicit evaluation sample sizes. Partial or incompatible caches fail without a silent refit.


In [ ]:
RUN_AGREEMENT = True
RUN_VALIDATION = True
RUN_SEMANTIC_ANALYSIS = os.environ.get("UKB_RUN_SEMANTIC_ANALYSIS", "1") != "0"
SEMANTIC_LOCAL_ONLY = True
RUN_VALIDATION_INFERENCE = os.environ.get("UKB_RUN_VALIDATION_INFERENCE", "0") == "1"

COMBINED_LABELS_CSV = os.environ.get("UKB_COMBINED_LABELS_CSV", "").strip() or None
VALIDATION_POSITIVE_CSV = os.environ.get("UKB_VALIDATION_POSITIVE_CSV", "").strip() or None
VALIDATION_NEGATIVE_CSV = os.environ.get("UKB_VALIDATION_NEGATIVE_CSV", "").strip() or None
VALIDATION_N_POS = os.environ.get("UKB_VALIDATION_N_POS", "").strip() or None
VALIDATION_N_NEG = os.environ.get("UKB_VALIDATION_N_NEG", "").strip() or None
VALIDATION_OUTPUT_DIR = Path(os.environ.get("UKB_VALIDATION_OUTPUT_DIR", "").strip() or P.OUTPUT / "validation").expanduser()
if not VALIDATION_OUTPUT_DIR.is_absolute():
    VALIDATION_OUTPUT_DIR = P.ROOT / VALIDATION_OUTPUT_DIR

TABLE_DIR = P.TABLE_DATA_ANALYSIS / "00_dataset"
FIGURE_DIR = P.FIG_DATA_ANALYSIS / "00_dataset"
MAX_TFIDF_PER_GROUP = 20_000
MAX_SEMANTIC_PER_GROUP = 3_000
SEED = 42


In [ ]:
import pandas as pd
from IPython.display import Image, display
from utils.shared_style import load_style
from utils import data_analysis_00_agreement as A
from utils import data_analysis_00_validation as V

STYLE = load_style("00_dataset")
section_results = []

def run_section(name, enabled, function, **kwargs):
    if not enabled:
        result = {"section": name, "status": "SKIP", "detail": "Disabled in notebook configuration."}
        print(f"[SKIP] {name}: disabled in configuration.")
    else:
        try:
            result = function(**kwargs)
            result.setdefault("section", name)
            result.setdefault("detail", result.get("reason", "Completed."))
        except Exception as error:
            detail = f"{type(error).__name__}: {str(error).splitlines()[0]}"
            print(f"[FAIL] {name}: {detail}")
            result = {"section": name, "status": "FAIL", "detail": detail}
    section_results.append(result)
    return result

def show_current_figure(result, stem):
    # Only show artifacts generated by this run, never an unrelated stale PNG.
    paths = result.get("figures", result.get("figure_files", []))
    matches = [Path(path) for path in paths
               if Path(path).suffix == ".png" and Path(path).stem in (stem, stem + "_incomplete")]
    if matches:
        display(Image(filename=str(matches[0]), width=1100))
        print(P.raw_path(matches[0]))
    else:
        print(f"[SKIP] {stem}: {result.get('detail', result.get('reason', 'Source data unavailable.'))}")

def show_agreement_table(name):
    table = agreement_result.get("table_frames", {}).get(name)
    if table is not None:
        display(table)
    else:
        print(f"[SKIP] {name}: agreement analysis unavailable.")


## 2. Compute candidate diagnostics and inspect the corpus

All deployed classifier labels are read from the saved CSV. This runs the original overview, model/consensus summaries, annual counts, explicit mentions, keyword profiles, TF-IDF contrasts and semantic diagnostic. Tables and figures are exported once; later cells display those exact results.

Tables: `output/tables/data_analysis/00_dataset/three_model_agreement/`.
Figures: `output/figures/data_analysis/00_dataset/three_model_agreement/`.


In [ ]:
agreement_result = run_section(
    "Three-model agreement", RUN_AGREEMENT, A.run_agreement,
    input_path=COMBINED_LABELS_CSV,
    output_dir=TABLE_DIR / "three_model_agreement",
    figure_dir=FIGURE_DIR / "three_model_agreement",
    run_semantic=RUN_SEMANTIC_ANALYSIS,
    max_tfidf_per_group=MAX_TFIDF_PER_GROUP,
    max_semantic_per_group=MAX_SEMANTIC_PER_GROUP,
    seed=SEED, show_tables=False, show_figures=False,
    semantic_local_only=SEMANTIC_LOCAL_ONLY,
)


In [ ]:
show_agreement_table("overview")
show_agreement_table("model_summary")


## 3. Restored manuscript figure: consensus diagnostics (2×2)

**A**, Annual TRUE predictions from Qwen, Llama and Mistral. **B**, Annual unanimous-TRUE versus remaining candidate counts (log scale). **C**, Keyword profiles in the two groups. **D**, The sentence-embedding projection coloured by consensus, with its recomputed silhouette index.

The same four analyses shown in the original manuscript panel are reunited here. The map describes separation of model-defined groups; it is not independent validation accuracy. If semantic data are unavailable, only an explicitly named `_incomplete` version is exported.


In [ ]:
show_current_figure(agreement_result, "00_06_figure_consensus_validation")
if "semantic_metrics" in agreement_result.get("table_frames", {}):
    show_agreement_table("semantic_metrics")


## 4. Model agreement, consensus categories and annual trends

The original vote, consensus-group and vote-signature tables distinguish unanimous FALSE, disagreements and unparsed responses. TRUE-vote counts alone do not distinguish those outcomes, so the consensus-group bar chart is also retained. Pairwise agreement uses only labels parsed by both models; its CSV supplies the corresponding denominator.


In [ ]:
show_current_figure(agreement_result, "00_01_figure_candidate_agreement")
show_agreement_table("pairwise_agreement")
show_agreement_table("yearly")


In [ ]:
show_current_figure(agreement_result, "00_07_figure_consensus_groups")
show_agreement_table("vote_distribution")
show_agreement_table("group_distribution")
show_agreement_table("signature_distribution")


## 5. Explicit mentions, keyword profiles and TF-IDF contrasts

The original explicit-mention table and full keyword counts are visible below. The text figure again shows up to **25 terms per direction**. The CSV retains all term scores; the tables below show the same leading terms with both group means and their difference.


In [ ]:
show_current_figure(agreement_result, "00_02_figure_candidate_text")
show_agreement_table("explicit_summary")
show_agreement_table("category_summary")
terms = agreement_result.get("table_frames", {}).get("tfidf_terms")
if terms is not None:
    difference = "difference_TRUE_minus_rest"
    display(terms.loc[terms[difference].gt(0)].nlargest(25, difference))
    display(terms.loc[terms[difference].lt(0)].nsmallest(25, difference))


## 6. Semantic diagnostic by group and publication year

Both original semantic views are preserved. They use the same sampled papers, coordinates and axes, with group labels on the left and publication year on the right. Sample identities, texts and years must match before cached coordinates can be reused. Metrics refer to the full embedding space; the figure is a two-dimensional projection.


In [ ]:
show_current_figure(agreement_result, "00_05_figure_semantic_diagnostics")


## 7. Validation inputs and saved model results

The original six-prompt validation analysis is retained below. If its files are absent, the notebook lists the exact inputs required and still completes the candidate analyses above. Supplying candidate labels does not supply these separate prompt-specific validation predictions.

**Encoder interpretation:** the original SciBERT and MiniLM baselines optimise their similarity thresholds on the evaluation labels. Their reported performance is in-sample calibration, not independent held-out validation.


In [ ]:
validation_availability = V.validation_cache_status(VALIDATION_OUTPUT_DIR)
display(pd.DataFrame(validation_availability["figure_status"]))
if validation_availability["missing_inputs"]:
    display(pd.DataFrame({"Missing validation input": [P.raw_path(path) for path in validation_availability["missing_inputs"]]}))


In [ ]:
validation_result = run_section(
    "Labelled validation", RUN_VALIDATION, V.run_validation,
    output_dir=VALIDATION_OUTPUT_DIR,
    positive_csv=VALIDATION_POSITIVE_CSV,
    negative_csv=VALIDATION_NEGATIVE_CSV,
    n_positive=VALIDATION_N_POS, n_negative=VALIDATION_N_NEG,
    run_inference=RUN_VALIDATION_INFERENCE,
    figure_dir=FIGURE_DIR / "validation",
    show_figures=False,
)


## 8. Original six-prompt pairwise-agreement figure (2×3)

**A–F**, Conservative, balanced, evidence-cue, context/no-shot, one-shot and five-shot prompts. Each matrix compares all available classifiers using papers with two parsed predictions. The original six-panel layout is retained, with shared colours, cell borders and paired-paper counts. Agreement measures consistency rather than accuracy against ground truth.


In [ ]:
show_current_figure(validation_result, "00_04_figure_validation_agreement")


## 9. Validation performance and model rankings (2×2)

**A–D**, Accuracy, precision, recall and F1 for each model and prompt. The full metrics table includes parse coverage and confusion-matrix counts. Rankings are shown separately; encoder rows remain explicitly labelled as in-sample results. Original runtime and throughput values are retained when present in saved results.


In [ ]:
show_current_figure(validation_result, "00_03_figure_validation_performance")
if validation_result.get("status") == "PASS":
    display(validation_result["summary"])
    display(validation_result["ranked"])


## 10. Output inventory and completion status

The inventory lists this run's actual tables and figures. Missing validation or semantic results stay visible as skipped/incomplete analyses; a successful notebook execution does not imply those results were available. Failures are raised after both analysis sections have been attempted.


In [ ]:
artifact_rows = []
for result in (agreement_result, validation_result):
    paths = [*result.get("tables", []), *result.get("figures", []), *result.get("files", [])]
    for path in dict.fromkeys(map(str, paths)):
        path = Path(path)
        if path.is_file():
            artifact_rows.append({"section": result["section"], "file": path.name,
                                  "path": P.raw_path(path), "size_kb": round(path.stat().st_size / 1024, 1)})
if artifact_rows:
    inventory = pd.DataFrame(artifact_rows)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    inventory.to_csv(TABLE_DIR / "00_dataset_artifact_inventory.csv", index=False)
    display(inventory)


In [ ]:
summary = pd.DataFrame([
    {"section": result["section"], "status": result["status"], "detail": result["detail"]}
    for result in section_results
])
if agreement_result.get("semantic_status") == "SKIP":
    summary.loc[len(summary)] = ["Semantic separation", "SKIP", "No matching semantic cache; encoding disabled."]
TABLE_DIR.mkdir(parents=True, exist_ok=True)
status_path = TABLE_DIR / "00_dataset_status.csv"
summary.to_csv(status_path, index=False)
display(summary)
counts = summary["status"].value_counts()
print(f"{counts.get('PASS', 0)} completed, {counts.get('SKIP', 0)} skipped, {counts.get('FAIL', 0)} failed.")
print("Status:", P.raw_path(status_path))
if summary["status"].eq("FAIL").any():
    raise RuntimeError("Dataset analysis failed: " + "; ".join(summary.loc[summary["status"].eq("FAIL"), "detail"]))


---

Part II starts from a clean namespace so its original model, table and figure names cannot inherit state from Part I.


In [ ]:
try:
    import matplotlib as _boundary_mpl
    import matplotlib.pyplot as _boundary_plt
    _boundary_plt.close("all")
    _boundary_mpl.rcdefaults()
except ImportError:
    pass
import warnings as _boundary_warnings
_boundary_warnings.resetwarnings()
from IPython import get_ipython as _boundary_get_ipython
_boundary_get_ipython().run_line_magic("reset", "-f")


# Part II: Locked development/held-out classifier validation

This part compares six prompt strategies across four instruction-tuned LLMs and two encoder baselines. It uses a fixed, reproducible development/held-out design so prompt selection and encoder-threshold calibration do not use the held-out labels.

**Inputs**

- `data/validation/ukb_ground_truth_positive_labelled.csv`: known UKB-use publications (`label=1`)
- `data/validation/ukb_negative_pre2013_labelled_final.csv`: pre-2013 negative baseline (`label=0`)
- `output/validation/ukb_prompt_validation_heldout_v3/tables/`: the frozen splits, predictions, metrics, model revisions and prompt-selection record

Both labelled files must contain `id`, `title`, `abstract`, `year`, and `label`.

**Workflow**

1. Reserve reproducible one-shot and five-shot examples outside the benchmark.
2. Sample 4,000 positives and 4,000 negatives using a fixed seed.
3. Split the benchmark into an 80% development set and a 20% held-out test set, stratified by label.
4. Compare all prompts on the development set and select one using a pre-specified mean-LLM F1 rule.
5. Calibrate SciBERT and MiniLM similarity thresholds using development data only.
6. Freeze the selected prompt and thresholds before held-out evaluation.
7. Run all prompt/model combinations on the held-out set for robustness plots; only the pre-selected prompt is used for the primary held-out performance claim.

The frozen development results select **prompt 2 (`p2_balanced`)**, the same prompt used for the production LLM-tagging task. Consequently, no downstream tagging analysis is changed. Regeneration mode verifies this decision from the development metrics before reading any held-out performance claim.

The default saved-results mode validates the supplied inputs and result bundle before rebuilding all figures. A full inference rerun is opt-in via `UKB_RERUN_HELDOUT_MODELS=1` and requires CUDA, the optional quantisation dependencies, model access, and `HF_TOKEN`.


In [ ]:
import ast
import gc
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "utils").is_dir()
)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from utils import shared_paths as P
from utils.shared_style import load_style

P.bootstrap()
STYLE = load_style("00_dataset")
warnings.filterwarnings("ignore", category=FutureWarning)


## Configuration

Canonical repository paths come from `utils.shared_paths`. Saved-results mode is the reproducible default and requires no model downloads. Set `UKB_RERUN_HELDOUT_MODELS=1` for an explicit inference rerun; the rerun fails early unless CUDA, the optional GPU dependencies and `HF_TOKEN` are available.


In [ ]:
POSITIVE_PATH = P.VALIDATION_POSITIVE
NEGATIVE_PATH = P.VALIDATION_NEGATIVE
OUT_DIR = P.VALIDATION_HELDOUT
INTERMEDIATE_DIR = P.VALIDATION_HELDOUT_INTERMEDIATE
TABLE_DIR = P.VALIDATION_HELDOUT_TABLES
FIGURE_DIR = P.VALIDATION_HELDOUT_FIGURES
for directory in (OUT_DIR, INTERMEDIATE_DIR, TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RERUN_MODELS = os.environ.get("UKB_RERUN_HELDOUT_MODELS", "0") == "1"
N_POS = 4_000
N_NEG = 4_000
HELDOUT_FRACTION = 0.20
SAMPLING_SEED = 42
EXPECTED_SELECTED_PROMPT = "p2_balanced"
MAX_INPUT_TOKENS = 4_096
MAX_NEW_TOKENS = 32
BATCH_SIZE = 8
MAX_PARSE_ATTEMPTS = 3
USE_4BIT = True

# Keep the original variable name for the retained plotting cells, but resolve it from
# the project's single shared palette rather than a notebook-local colour scheme.
LCDS_PALETTE = list(STYLE["colors"][:6])

LLM_SPECS = [
    ("Qwen/Qwen2.5-7B-Instruct", "qwen2_5_7b", "Qwen2.5-7B"),
    ("meta-llama/Meta-Llama-3-8B-Instruct", "llama3_8b", "Llama-3-8B"),
    ("mistralai/Mistral-7B-Instruct-v0.3", "mistral_7b", "Mistral-7B"),
    ("HuggingFaceH4/zephyr-7b-beta", "zephyr_7b", "Zephyr-7B"),
]
ENCODER_SPECS = [
    ("allenai/scibert_scivocab_uncased", "scibert_sim", "SciBERT"),
    ("sentence-transformers/all-MiniLM-L6-v2", "sbert_minilm_sim", "MiniLM"),
]
MODEL_TAGS = [item[1] for item in LLM_SPECS + ENCODER_SPECS]
MODEL_LABELS = {item[1]: item[2] for item in LLM_SPECS + ENCODER_SPECS}

random.seed(SAMPLING_SEED)
np.random.seed(SAMPLING_SEED)

if RERUN_MODELS:
    try:
        import bitsandbytes as bnb
        import torch
        from huggingface_hub import HfApi, login
        from sentence_transformers import SentenceTransformer
        from transformers import (
            AutoModel,
            AutoModelForCausalLM,
            AutoTokenizer,
            BitsAndBytesConfig,
        )
    except ImportError as exc:
        raise RuntimeError(
            "Full held-out inference requires the optional GPU/model dependencies. "
            "Install requirements-analysis.txt plus bitsandbytes."
        ) from exc
    if not torch.cuda.is_available():
        raise RuntimeError("UKB_RERUN_HELDOUT_MODELS=1 requires a CUDA GPU.")

    torch.manual_seed(SAMPLING_SEED)
    torch.cuda.manual_seed_all(SAMPLING_SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

    hf_token = os.environ.get("HF_TOKEN", "").strip()
    if not hf_token:
        raise ValueError("HF_TOKEN is required for the complete six-model rerun.")
    login(token=hf_token, add_to_git_credential=False)
    gpu_name = torch.cuda.get_device_name(0)
    COMPUTE_DTYPE = torch.float16
    print("Mode: full model rerun")
    print("GPU:", gpu_name)
    print("bitsandbytes:", bnb.__version__)
    print("Compute dtype:", COMPUTE_DTYPE)
else:
    hf_token = None
    gpu_name = "not used (saved-results mode)"
    COMPUTE_DTYPE = None
    print("Mode: saved-results figure regeneration")

print("Output directory:", P.raw_path(OUT_DIR))


## Input preparation and locked split

The demonstration papers are reserved first and cannot enter either evaluation split. Sorting by ID before seeded sampling makes the selection insensitive to source row order.

In [ ]:
REQUIRED_COLUMNS = ["id", "title", "abstract", "year", "label"]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_text(value):
    if pd.isna(value):
        return ""
    if not isinstance(value, str):
        return str(value).strip()
    text = value.strip()
    if text.startswith("{") and text.endswith("}"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, dict):
                if parsed.get("preferred") is not None:
                    return str(parsed["preferred"]).strip()
                for candidate in parsed.values():
                    if candidate is not None and str(candidate).strip():
                        return str(candidate).strip()
        except Exception:
            pass
    return text


def load_labelled(path, expected_label):
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, low_memory=False)
    missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")
    frame = frame[REQUIRED_COLUMNS].copy()
    frame["id"] = frame["id"].astype(str).str.strip()
    frame["title"] = frame["title"].apply(extract_text)
    frame["abstract"] = frame["abstract"].apply(extract_text)
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
    frame["label"] = int(expected_label)
    frame = frame[frame["id"].ne("")]
    frame = frame[frame["abstract"].str.strip().ne("")]
    frame = frame.drop_duplicates("id", keep="first")
    return frame.sort_values("id").reset_index(drop=True)


positive_pool = load_labelled(POSITIVE_PATH, 1)
negative_pool = load_labelled(NEGATIVE_PATH, 0)
overlap = set(positive_pool["id"]) & set(negative_pool["id"])
if overlap:
    raise ValueError(f"Positive and negative inputs overlap on {len(overlap)} IDs.")


def contains_explicit_ukb(row):
    text = f"{row['title']} {row['abstract']}".lower()
    return any(term in text for term in [
        "uk biobank", "ukb", "ukbb", "united kingdom biobank",
    ])


def contains_hard_negative_cue(row):
    text = f"{row['title']} {row['abstract']}".lower()
    return any(term in text for term in [
        "unlike uk biobank", "compared with uk biobank",
        "such as uk biobank", "including uk biobank",
        "biobanks such as", "ethical", "governance",
        "china kadoorie", "biobank japan", "finngen",
        "all of us", "janus serum bank",
    ])


def reproducible_choice(preferred, fallback, n, seed, excluded=None):
    excluded = set() if excluded is None else set(excluded)
    preferred = preferred[~preferred["id"].isin(excluded)].sort_values("id")
    fallback = fallback[~fallback["id"].isin(excluded)].sort_values("id")
    selected = preferred.sample(n=min(n, len(preferred)), random_state=seed)
    if len(selected) < n:
        needed = n - len(selected)
        remainder = fallback[~fallback["id"].isin(selected["id"])]
        selected = pd.concat([
            selected,
            remainder.sample(n=needed, random_state=seed + 100),
        ])
    return selected.sort_values("id").reset_index(drop=True)


if RERUN_MODELS:
    positive_no_explicit = positive_pool[
        ~positive_pool.apply(contains_explicit_ukb, axis=1)
    ]
    negative_hard = negative_pool[
        negative_pool.apply(contains_hard_negative_cue, axis=1)
    ]
    demo_positive = reproducible_choice(
        positive_no_explicit, positive_pool, 3, SAMPLING_SEED + 1
    )
    demo_negative = reproducible_choice(
        negative_hard, negative_pool, 2, SAMPLING_SEED + 2
    )
    demonstrations = pd.concat([demo_positive, demo_negative], ignore_index=True)
    demonstrations["demo_role"] = [
        "one-shot and five-shot", "five-shot", "five-shot",
        "one-shot and five-shot", "five-shot",
    ]
    demonstrations.to_csv(TABLE_DIR / "few_shot_examples.csv", index=False)

    demo_ids = set(demonstrations["id"])
    positive_available = positive_pool[~positive_pool["id"].isin(demo_ids)]
    negative_available = negative_pool[~negative_pool["id"].isin(demo_ids)]
    if len(positive_available) < N_POS or len(negative_available) < N_NEG:
        raise ValueError(
            "Insufficient records after reserving demonstrations: "
            f"positive={len(positive_available)}, negative={len(negative_available)}"
        )
    sampled_positive = positive_available.sample(n=N_POS, random_state=SAMPLING_SEED)
    sampled_negative = negative_available.sample(n=N_NEG, random_state=SAMPLING_SEED)
    benchmark = pd.concat([sampled_positive, sampled_negative], ignore_index=True)
    benchmark = benchmark.sort_values("id").reset_index(drop=True)
    development, heldout = train_test_split(
        benchmark,
        test_size=HELDOUT_FRACTION,
        random_state=SAMPLING_SEED,
        stratify=benchmark["label"],
    )
    development = development.sort_values("id").reset_index(drop=True)
    heldout = heldout.sort_values("id").reset_index(drop=True)
    development["split"] = "development"
    heldout["split"] = "heldout"
    development.to_csv(TABLE_DIR / "benchmark_development.csv", index=False)
    heldout.to_csv(TABLE_DIR / "benchmark_heldout_test.csv", index=False)

    input_manifest = pd.DataFrame([
        {
            "role": "positive", "path": str(POSITIVE_PATH),
            "sha256": sha256_file(POSITIVE_PATH), "rows": len(positive_pool),
            "unique_ids": positive_pool["id"].nunique(),
        },
        {
            "role": "negative", "path": str(NEGATIVE_PATH),
            "sha256": sha256_file(NEGATIVE_PATH), "rows": len(negative_pool),
            "unique_ids": negative_pool["id"].nunique(),
        },
    ])
    input_manifest.to_csv(TABLE_DIR / "input_manifest.csv", index=False)
else:
    required_saved = [
        "benchmark_development.csv", "benchmark_heldout_test.csv",
        "few_shot_examples.csv", "input_manifest.csv",
    ]
    missing_saved = [name for name in required_saved if not (TABLE_DIR / name).is_file()]
    if missing_saved:
        raise FileNotFoundError(
            "Saved-results mode is missing required tables: " + ", ".join(missing_saved)
        )

    development = pd.read_csv(TABLE_DIR / "benchmark_development.csv", low_memory=False)
    heldout = pd.read_csv(TABLE_DIR / "benchmark_heldout_test.csv", low_memory=False)
    demonstrations = pd.read_csv(TABLE_DIR / "few_shot_examples.csv", low_memory=False)
    expected_class_counts = {
        "development": {0: N_NEG - int(N_NEG * HELDOUT_FRACTION),
                        1: N_POS - int(N_POS * HELDOUT_FRACTION)},
        "heldout": {0: int(N_NEG * HELDOUT_FRACTION),
                    1: int(N_POS * HELDOUT_FRACTION)},
    }
    for name, frame, expected_split in [
        ("development", development, "development"),
        ("heldout", heldout, "heldout"),
    ]:
        missing = sorted(set(REQUIRED_COLUMNS + ["split"]) - set(frame.columns))
        if missing:
            raise ValueError(f"Saved {name} benchmark is missing columns: {missing}")
        frame["id"] = frame["id"].astype(str).str.strip()
        frame["label"] = pd.to_numeric(frame["label"], errors="raise").astype(int)
        if set(frame["split"]) != {expected_split}:
            raise ValueError(f"Saved {name} benchmark has an invalid split column.")
        if not frame["id"].is_unique:
            raise ValueError(f"Saved {name} benchmark contains duplicate IDs.")
        class_counts = frame["label"].value_counts().sort_index().to_dict()
        if class_counts != expected_class_counts[name]:
            raise ValueError(
                f"Saved {name} class counts are {class_counts}; expected "
                f"{expected_class_counts[name]} for the locked 4,000/4,000 benchmark."
            )
    if set(development["id"]) & set(heldout["id"]):
        raise ValueError("Saved development and held-out benchmarks overlap.")

    demonstrations["id"] = demonstrations["id"].astype(str).str.strip()
    demonstrations["label"] = pd.to_numeric(
        demonstrations["label"], errors="raise"
    ).astype(int)
    if not demonstrations["id"].is_unique:
        raise ValueError("Saved few-shot examples contain duplicate IDs.")
    demo_positive = demonstrations[demonstrations["label"].eq(1)].reset_index(drop=True)
    demo_negative = demonstrations[demonstrations["label"].eq(0)].reset_index(drop=True)
    if len(demo_positive) != 3 or len(demo_negative) != 2:
        raise ValueError("Saved few-shot examples must contain three positives and two negatives.")
    if set(demonstrations["id"].astype(str)) & set(pd.concat([development, heldout])["id"]):
        raise ValueError("A saved demonstration record appears in an evaluation split.")

    source_labels = pd.concat([
        positive_pool[["id", "label"]], negative_pool[["id", "label"]]
    ]).set_index("id")["label"]
    benchmark = pd.concat([development, heldout], ignore_index=True)
    if not benchmark["id"].isin(source_labels.index).all():
        raise ValueError("Saved benchmark contains IDs absent from the labelled inputs.")
    expected_labels = benchmark["id"].map(source_labels).astype(int)
    if not expected_labels.eq(benchmark["label"].astype(int)).all():
        raise ValueError("Saved benchmark labels disagree with the labelled inputs.")

    input_manifest = pd.read_csv(TABLE_DIR / "input_manifest.csv")
    manifest_by_role = input_manifest.set_index("role")
    for role, path, pool in [
        ("positive", POSITIVE_PATH, positive_pool),
        ("negative", NEGATIVE_PATH, negative_pool),
    ]:
        if role not in manifest_by_role.index:
            raise ValueError(f"Input manifest has no {role!r} row.")
        manifest_row = manifest_by_role.loc[role]
        if sha256_file(path) != str(manifest_row["sha256"]):
            raise ValueError(f"{path.name} does not match the recorded input hash.")
        if len(pool) != int(manifest_row["rows"]):
            raise ValueError(f"{path.name} does not match the recorded row count.")

pd.concat([development, heldout], ignore_index=True).to_csv(
    TABLE_DIR / "benchmark_with_split.csv", index=False
)
split_summary = (
    pd.concat([development, heldout])
    .groupby(["split", "label"], as_index=False)
    .size()
)
print("Positive pool:", len(positive_pool))
print("Negative pool:", len(negative_pool))
display(demonstrations[["id", "year", "label", "demo_role"]])
display(split_summary)


## Prompt definitions

The six prompt conditions are retained verbatim in the saved notebook. Every LLM receives the same paper text and decoding budget within a condition.

In [ ]:
one_positive = demo_positive.iloc[0]
one_negative = demo_negative.iloc[0]
five_shot_examples = [
    *demo_positive.assign(label=True).to_dict("records"),
    *demo_negative.assign(label=False).to_dict("records"),
]
random.Random(SAMPLING_SEED).shuffle(five_shot_examples)


def truncate_text(value, limit):
    return str(value or "").strip()[:limit]


def json_instruction():
    return 'Return exactly one JSON object: {"implies_UKB_use": true} or {"implies_UKB_use": false}.'


def make_prompt_v1_conservative(title, abstract):
    return f"""You will be given a scientific paper title and abstract.

Task: decide whether the paper used UK Biobank data or resources.

Definition:
- true: the study used UK Biobank data, participants, samples, imaging, genetics, linked health records, or another UK Biobank resource.
- false: the paper only mentions UK Biobank, discusses biobanks generally, compares with UK Biobank, or uses other biobanks but not UK Biobank.

Be conservative. If uncertain, return false.

{json_instruction()}

title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v2_balanced(title, abstract):
    return f"""You will be given a scientific paper title and abstract.

Task: decide whether the paper likely used UK Biobank data or resources.

Important:
- Some true UK Biobank-use papers do not mention "UK Biobank" in the abstract.
- The paper may still use UK Biobank if the abstract describes a UK population-scale cohort, genetic/imaging/health-record analysis, or data-resource use that is consistent with UK Biobank.
- Do not require explicit words "UK Biobank" if the evidence strongly suggests use.

Return true when the title/abstract provides reasonable evidence that the paper analysed UK Biobank participants, data, samples, imaging, genetics, linked records, or a UK Biobank-derived cohort.
Return false when the paper is only about generic biobanking, ethics/governance, reviews, comparisons, or another named biobank.

{json_instruction()}
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v3_evidence_cues(title, abstract):
    return f"""Classify whether this paper uses UK Biobank.

Use the following cues.

Positive evidence can include:
- explicit UK Biobank / UKB / UKBB mention;
- analysis of a very large UK cohort with genetic, imaging, health-record, lifestyle, biomarker, or hospital-linked data;
- phrases such as participants, cohort, baseline assessment, imaging assessment, genotyping, exome sequencing, linked health records, Hospital Episode Statistics, or Townsend deprivation index in a UK population context;
- a study design that clearly analyses participant-level data rather than merely discussing biobanks.

Negative evidence can include:
- generic discussion of biobanks;
- ethics, governance, consent, infrastructure, sample storage, or review articles;
- use of another biobank only;
- mentions like "such as UK Biobank", "unlike UK Biobank", or comparison with UK Biobank.

Prefer true if the abstract strongly looks like an original analysis using UK Biobank-style data, even if UK Biobank is not named in the abstract.
Prefer false if the abstract is generic or only about other biobanks.

{json_instruction()}
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v4_context_no_shot(title, abstract):
    return f"""You will classify a paper using only its title and abstract.

Context:
All papers in this evaluation were retrieved because their full text matched at least one UK Biobank-related query. Therefore, UK Biobank may be mentioned only outside the abstract.

Question:
Based on the title and abstract, is it likely that the paper used UK Biobank data/resources in its own analysis?

Label true if likely UK Biobank use.
Label false if the paper likely only mentions UK Biobank, discusses biobanks generally, or uses other non-UKB resources.

Do not be overly strict: if the paper is an original epidemiological, genetic, imaging, biomarker, or clinical-risk study and the abstract strongly suggests use of a UK population-scale linked cohort, return true.

{json_instruction()}
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v5_one_shot(title, abstract):
    return f"""You will be given a scientific paper title and abstract.

Task:
Decide whether the paper likely used UK Biobank data/resources in its own analysis.

Context:
All papers in this evaluation were retrieved because their full text matched a UK Biobank-related search. Some true positives may not mention UK Biobank in the abstract.

{json_instruction()}

Example positive:
title: \"\"\"{truncate_text(one_positive['title'], 500)}\"\"\"
abstract: \"\"\"{truncate_text(one_positive['abstract'], 1800)}\"\"\"
answer: {{"implies_UKB_use":true}}

Example negative:
title: \"\"\"{truncate_text(one_negative['title'], 500)}\"\"\"
abstract: \"\"\"{truncate_text(one_negative['abstract'], 1800)}\"\"\"
answer: {{"implies_UKB_use":false}}

Now classify this paper.
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v6_five_shot(title, abstract):
    blocks = []
    for index, example in enumerate(five_shot_examples, start=1):
        answer = str(bool(example["label"])).lower()
        blocks.append(f"""Example {index}
title: \"\"\"{truncate_text(example['title'], 450)}\"\"\"
abstract: \"\"\"{truncate_text(example['abstract'], 1300)}\"\"\"
answer: {{"implies_UKB_use":{answer}}}""")
    examples = "\n\n".join(blocks)
    return f"""You will be given a scientific paper title and abstract.

Task:
Decide whether the paper likely used UK Biobank data/resources in its own analysis.

Context:
All papers in this evaluation were retrieved because their full text matched a UK Biobank-related search.
Some true UK Biobank-use papers may not mention UK Biobank in the abstract.
Some false positives mention biobanks, UK cohorts, or other biobanks but do not use UK Biobank.

Return true when the paper likely analysed UK Biobank participants, data, samples, imaging, genetics, linked health records, or other UK Biobank resources.
Return false for generic biobank discussion, ethics/governance, reviews, comparisons, or other-biobank-only papers.

{json_instruction()}

Few-shot examples:
{examples}

Now classify this paper.
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


PROMPT_BUILDERS = {
    "p1_conservative": make_prompt_v1_conservative,
    "p2_balanced": make_prompt_v2_balanced,
    "p3_evidence_cues": make_prompt_v3_evidence_cues,
    "p4_context_no_shot": make_prompt_v4_context_no_shot,
    "p5_real_one_shot": make_prompt_v5_one_shot,
    "p6_real_five_shot": make_prompt_v6_five_shot,
}
PROMPT_LABELS = {
    "p1_conservative": "Conservative",
    "p2_balanced": "Balanced",
    "p3_evidence_cues": "Evidence cues",
    "p4_context_no_shot": "Context, no-shot",
    "p5_real_one_shot": "One-shot",
    "p6_real_five_shot": "Five-shot",
}

expected_prompt_manifest = pd.DataFrame([
    {"prompt": name, "display_name": PROMPT_LABELS[name]}
    for name in PROMPT_BUILDERS
])
prompt_manifest_path = TABLE_DIR / "prompt_manifest.csv"
if RERUN_MODELS:
    prompt_manifest = expected_prompt_manifest
    prompt_manifest.to_csv(prompt_manifest_path, index=False)
else:
    if not prompt_manifest_path.is_file():
        raise FileNotFoundError(prompt_manifest_path)
    prompt_manifest = pd.read_csv(prompt_manifest_path)
    pd.testing.assert_frame_equal(
        prompt_manifest.reset_index(drop=True),
        expected_prompt_manifest.reset_index(drop=True),
        check_dtype=False,
    )
display(prompt_manifest)

## Resolve and record model revisions

Full-rerun mode resolves and records each repository commit before loading any model. Saved-results mode instead reads the immutable revisions and environment record from the supplied result bundle.


In [ ]:
if RERUN_MODELS:
    api = HfApi(token=hf_token)
    resolved_models = []
    for model_id, tag, display_name in LLM_SPECS + ENCODER_SPECS:
        info = api.model_info(model_id)
        resolved_models.append({
            "model_id": model_id,
            "tag": tag,
            "display_name": display_name,
            "revision": info.sha,
        })
    resolved_models = pd.DataFrame(resolved_models)
    resolved_models.to_csv(TABLE_DIR / "resolved_model_revisions.csv", index=False)
    REVISION = dict(zip(resolved_models["tag"], resolved_models["revision"]))

    packages = [
        "torch", "transformers", "accelerate", "bitsandbytes",
        "sentence-transformers", "huggingface-hub", "scikit-learn",
        "pandas", "numpy", "matplotlib", "seaborn",
    ]
    environment = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "gpu": gpu_name,
        "cuda": torch.version.cuda,
        "compute_dtype": str(COMPUTE_DTYPE),
        "sampling_seed": SAMPLING_SEED,
        "n_positive": N_POS,
        "n_negative": N_NEG,
        "heldout_fraction": HELDOUT_FRACTION,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "batch_size": BATCH_SIZE,
        "max_parse_attempts": MAX_PARSE_ATTEMPTS,
        "decoding": "greedy; do_sample=False; num_beams=1",
        "quantization": "4-bit NF4 with double quantization" if USE_4BIT else "none",
    }
    environment["packages"] = {
        package: importlib.metadata.version(package) for package in packages
    }
    with open(TABLE_DIR / "run_configuration.json", "w") as handle:
        json.dump(environment, handle, indent=2)
else:
    revision_path = TABLE_DIR / "resolved_model_revisions.csv"
    configuration_path = TABLE_DIR / "run_configuration.json"
    if not revision_path.is_file() or not configuration_path.is_file():
        raise FileNotFoundError(
            "Saved-results mode requires resolved_model_revisions.csv and "
            "run_configuration.json."
        )
    resolved_models = pd.read_csv(revision_path)
    if not resolved_models["tag"].is_unique:
        raise ValueError("Model revision table contains duplicate tags.")
    missing_tags = sorted(set(MODEL_TAGS) - set(resolved_models["tag"]))
    extra_tags = sorted(set(resolved_models["tag"]) - set(MODEL_TAGS))
    if missing_tags or extra_tags:
        raise ValueError(
            f"Model revision tags disagree with the analysis: "
            f"missing={missing_tags}, extra={extra_tags}"
        )
    if resolved_models["revision"].isna().any():
        raise ValueError("Model revision table contains an empty revision.")
    REVISION = dict(zip(resolved_models["tag"], resolved_models["revision"]))
    with open(configuration_path) as handle:
        environment = json.load(handle)
    expected_configuration = {
        "sampling_seed": SAMPLING_SEED,
        "n_positive": N_POS,
        "n_negative": N_NEG,
        "heldout_fraction": HELDOUT_FRACTION,
    }
    for key, expected in expected_configuration.items():
        observed = environment.get(key)
        matches = (
            np.isclose(float(observed), float(expected))
            if observed is not None else False
        )
        if not matches:
            raise ValueError(
                f"Run configuration has {key}={observed!r}; expected {expected!r}."
            )
    print("Recorded inference environment:", environment.get("gpu", "unknown GPU"))

display(resolved_models)


## Shared inference and evaluation functions

In [ ]:
def parse_binary_output(text):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", str(text).strip(), flags=re.I | re.S)
    candidates = [cleaned]
    match = re.search(r"\{.*?\}", cleaned, flags=re.S)
    if match:
        candidates.append(match.group(0))
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
            value = parsed.get("implies_UKB_use") if isinstance(parsed, dict) else None
            if isinstance(value, bool):
                return value
            if str(value).strip().lower() in {"true", "yes", "1"}:
                return True
            if str(value).strip().lower() in {"false", "no", "0"}:
                return False
        except Exception:
            pass
    direct = re.fullmatch(r"\s*(true|false|yes|no|positive|negative)\s*[.!]?\s*", cleaned, flags=re.I)
    if direct:
        return direct.group(1).lower() in {"true", "yes", "positive"}
    keyed = re.search(r'implies_UKB_use["\s:]+(true|false)', cleaned, flags=re.I)
    return keyed.group(1).lower() == "true" if keyed else None


def model_input(tokenizer, prompt):
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    return prompt


def load_llm(model_id, tag):
    revision = REVISION[tag]
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, revision=revision, token=hf_token, use_fast=True
    )
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = None
    if USE_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=revision,
        token=hf_token,
        device_map="auto",
        torch_dtype=COMPUTE_DTYPE,
        quantization_config=quantization_config,
        low_cpu_mem_usage=True,
    ).eval()
    return tokenizer, model


def cached_prediction_is_valid(path, frame):
    if not path.exists():
        return False
    try:
        cached = pd.read_csv(path, usecols=["id"])
        return cached["id"].astype(str).tolist() == frame["id"].astype(str).tolist()
    except Exception:
        return False


def run_llm_prompt(model, tokenizer, model_tag, prompt_name, prompt_fn, split_name, frame):
    prediction_path = INTERMEDIATE_DIR / f"predictions_{split_name}_{prompt_name}_{model_tag}.csv"
    raw_path = INTERMEDIATE_DIR / f"raw_outputs_{split_name}_{prompt_name}_{model_tag}.csv"
    if cached_prediction_is_valid(prediction_path, frame) and raw_path.exists():
        print("Using cache:", prediction_path.name)
        return pd.read_csv(prediction_path)

    predictions = [None] * len(frame)
    attempts_used = np.zeros(len(frame), dtype=int)
    raw_attempts = [[] for _ in range(len(frame))]
    unresolved = list(range(len(frame)))
    started = time.time()

    for attempt in range(1, MAX_PARSE_ATTEMPTS + 1):
        if not unresolved:
            break
        next_unresolved = []
        for start in tqdm(
            range(0, len(unresolved), BATCH_SIZE),
            desc=f"{split_name}:{prompt_name}:{model_tag}:attempt{attempt}",
        ):
            indices = unresolved[start:start + BATCH_SIZE]
            prompts = []
            for index in indices:
                row = frame.iloc[index]
                prompt = prompt_fn(row["title"], row["abstract"])
                if attempt > 1:
                    prompt += "\nYour previous response was not valid. Return only the requested JSON object."
                prompts.append(model_input(tokenizer, prompt))

            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_TOKENS,
            ).to(model.device)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    num_beams=1,
                    use_cache=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            new_tokens = generated[:, inputs["input_ids"].shape[1]:]
            outputs = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

            for index, output in zip(indices, outputs):
                raw_attempts[index].append(output)
                attempts_used[index] = attempt
                parsed = parse_binary_output(output)
                if parsed is None:
                    next_unresolved.append(index)
                else:
                    predictions[index] = bool(parsed)
            del inputs, generated, new_tokens
        unresolved = next_unresolved

    runtime = time.time() - started
    result = frame[["id", "label"]].copy()
    result["prediction"] = pd.array(predictions, dtype="boolean")
    result["parse_ok"] = result["prediction"].notna()
    result["attempts_used"] = attempts_used
    result["runtime_seconds"] = runtime
    result.to_csv(prediction_path, index=False)

    raw = frame[["id", "label"]].copy()
    raw["raw_attempts_json"] = [json.dumps(values) for values in raw_attempts]
    raw.to_csv(raw_path, index=False)
    return result


def run_llm_stage(split_name, frame):
    for model_id, model_tag, display_name in LLM_SPECS:
        print(f"\nLoading {display_name}: {model_id}@{REVISION[model_tag]}")
        tokenizer, model = load_llm(model_id, model_tag)
        for prompt_name, prompt_fn in PROMPT_BUILDERS.items():
            run_llm_prompt(
                model, tokenizer, model_tag, prompt_name,
                prompt_fn, split_name, frame,
            )
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()


def metric_row(true_values, predictions, parse_ok, **metadata):
    truth = np.asarray(true_values, dtype=bool)
    parsed = np.asarray(parse_ok, dtype=bool)
    effective = pd.Series(predictions).fillna(False).astype(bool).to_numpy()
    tn, fp, fn, tp = confusion_matrix(truth, effective, labels=[False, True]).ravel()
    return {
        **metadata,
        "n": len(truth),
        "n_parsed": int(parsed.sum()),
        "parse_rate": float(parsed.mean()),
        "accuracy": accuracy_score(truth, effective),
        "precision": precision_score(truth, effective, zero_division=0),
        "recall": recall_score(truth, effective, zero_division=0),
        "f1": f1_score(truth, effective, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def best_f1_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    valid = np.arange(len(thresholds))
    best = sorted(
        valid,
        key=lambda i: (f1[i], precision[i], recall[i], thresholds[i]),
        reverse=True,
    )[0]
    return float(thresholds[best]), float(f1[best])

## Development-stage LLM predictions

This is the only stage used to select the prompt.

In [ ]:
if RERUN_MODELS:
    run_llm_stage("development", development)
else:
    print("Development inference skipped; using the frozen result bundle.")


## Encoder similarities and development-only calibration

The encoder baselines are prompt-independent. Their decision thresholds are selected on development data and then applied unchanged to held-out data.

In [ ]:
def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


def scibert_scores(frame, split_name):
    path = INTERMEDIATE_DIR / f"scores_{split_name}_scibert_sim.csv"
    if path.exists() and cached_prediction_is_valid(path, frame):
        return pd.read_csv(path)
    model_id, tag, _ = ENCODER_SPECS[0]
    revision = REVISION[tag]
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
    model = AutoModel.from_pretrained(
        model_id, revision=revision, torch_dtype=COMPUTE_DTYPE
    ).to("cuda").eval()
    query = "This scientific paper uses UK Biobank data or resources."

    def encode(texts, batch_size=64):
        outputs = []
        for start in tqdm(range(0, len(texts), batch_size), desc=f"SciBERT:{split_name}"):
            batch = tokenizer(
                texts[start:start + batch_size], return_tensors="pt",
                padding=True, truncation=True, max_length=512,
            ).to("cuda")
            with torch.inference_mode():
                hidden = model(**batch).last_hidden_state
                pooled = mean_pool(hidden, batch["attention_mask"])
                pooled = torch.nn.functional.normalize(pooled, dim=1)
            outputs.append(pooled.float().cpu().numpy())
        return np.vstack(outputs)

    query_embedding = encode([query])[0]
    texts = (frame["title"] + "\n" + frame["abstract"]).tolist()
    embeddings = encode(texts)
    result = frame[["id", "label"]].copy()
    result["score"] = embeddings @ query_embedding
    result.to_csv(path, index=False)
    del model, tokenizer, embeddings
    gc.collect()
    torch.cuda.empty_cache()
    return result


def minilm_scores(frame, split_name):
    path = INTERMEDIATE_DIR / f"scores_{split_name}_sbert_minilm_sim.csv"
    if path.exists() and cached_prediction_is_valid(path, frame):
        return pd.read_csv(path)
    model_id, tag, _ = ENCODER_SPECS[1]
    model = SentenceTransformer(
        model_id, revision=REVISION[tag], device="cuda"
    )
    query = "This scientific paper uses UK Biobank data or resources."
    texts = (frame["title"] + "\n" + frame["abstract"]).tolist()
    query_embedding = model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    embeddings = model.encode(
        texts, normalize_embeddings=True, convert_to_numpy=True,
        batch_size=128, show_progress_bar=True,
    )
    result = frame[["id", "label"]].copy()
    result["score"] = embeddings @ query_embedding
    result.to_csv(path, index=False)
    del model, embeddings
    gc.collect()
    torch.cuda.empty_cache()
    return result


def summarise_prompt_selection(metrics):
    summary = (
        metrics[metrics["model_type"].eq("LLM")]
        .groupby("prompt", as_index=False)
        .agg(
            mean_llm_f1=("f1", "mean"),
            mean_llm_precision=("precision", "mean"),
            mean_llm_recall=("recall", "mean"),
            mean_llm_accuracy=("accuracy", "mean"),
            minimum_parse_rate=("parse_rate", "min"),
        )
        .sort_values(
            ["mean_llm_f1", "mean_llm_precision", "mean_llm_recall", "prompt"],
            ascending=[False, False, False, True],
        )
        .reset_index(drop=True)
    )
    summary["selected"] = summary.index == 0
    return summary


if RERUN_MODELS:
    dev_encoder_scores = {
        "scibert_sim": scibert_scores(development, "development"),
        "sbert_minilm_sim": minilm_scores(development, "development"),
    }
    threshold_rows = []
    for tag, score_frame in dev_encoder_scores.items():
        threshold, development_f1 = best_f1_threshold(
            score_frame["label"].astype(bool).to_numpy(),
            score_frame["score"].to_numpy(),
        )
        threshold_rows.append({
            "model": tag,
            "threshold": threshold,
            "development_f1_at_threshold": development_f1,
        })
    encoder_thresholds = pd.DataFrame(threshold_rows)
    encoder_thresholds.to_csv(TABLE_DIR / "encoder_thresholds_development.csv", index=False)
    THRESHOLD = dict(zip(encoder_thresholds["model"], encoder_thresholds["threshold"]))
    display(encoder_thresholds)

else:
    threshold_path = TABLE_DIR / "encoder_thresholds_development.csv"
    if not threshold_path.is_file():
        raise FileNotFoundError(threshold_path)
    encoder_thresholds = pd.read_csv(threshold_path)
    if set(encoder_thresholds["model"]) != {item[1] for item in ENCODER_SPECS}:
        raise ValueError("Saved encoder threshold table does not match ENCODER_SPECS.")
    THRESHOLD = dict(zip(encoder_thresholds["model"], encoder_thresholds["threshold"]))
    dev_encoder_scores = None
    display(encoder_thresholds)


## Development metrics and prompt lock

In [ ]:
def read_llm_prediction(split_name, prompt_name, model_tag):
    path = INTERMEDIATE_DIR / f"predictions_{split_name}_{prompt_name}_{model_tag}.csv"
    result = pd.read_csv(path)
    result["prediction"] = result["prediction"].map(
        {True: True, False: False, "True": True, "False": False}
    )
    result["parse_ok"] = result["parse_ok"].astype(str).str.lower().eq("true")
    return result


def collect_metrics(split_name, frame, encoder_scores):
    rows = []
    for prompt_name in PROMPT_BUILDERS:
        for _, tag, _ in LLM_SPECS:
            prediction = read_llm_prediction(split_name, prompt_name, tag)
            rows.append(metric_row(
                prediction["label"], prediction["prediction"], prediction["parse_ok"],
                split=split_name, prompt=prompt_name, model=tag, model_type="LLM",
            ))
        for tag, score_frame in encoder_scores.items():
            prediction = score_frame["score"].ge(THRESHOLD[tag])
            rows.append(metric_row(
                score_frame["label"], prediction, np.ones(len(score_frame), dtype=bool),
                split=split_name, prompt=prompt_name, model=tag,
                model_type="encoder baseline",
            ))
    return pd.DataFrame(rows)



if RERUN_MODELS:
    development_metrics = collect_metrics(
        "development", development, dev_encoder_scores
    )
    development_metrics.to_csv(TABLE_DIR / "development_metrics_all_prompts_models.csv", index=False)

    prompt_selection = summarise_prompt_selection(development_metrics)
    SELECTED_PROMPT = prompt_selection.loc[0, "prompt"]
    if SELECTED_PROMPT != EXPECTED_SELECTED_PROMPT:
        raise RuntimeError(
            f"Development selection changed to {SELECTED_PROMPT}; expected "
            f"{EXPECTED_SELECTED_PROMPT}, the prompt used for production tagging."
        )
    prompt_selection.to_csv(TABLE_DIR / "development_prompt_selection.csv", index=False)

    prompt_lock = {
        "selected_prompt": SELECTED_PROMPT,
        "selection_rule": (
            "Highest mean development F1 across the four instruction-tuned LLMs; "
            "ties broken by mean precision, mean recall, then prompt name."
        ),
        "sampling_seed": SAMPLING_SEED,
        "encoder_thresholds": THRESHOLD,
    }
    with open(TABLE_DIR / "selected_prompt_before_heldout.json", "w") as handle:
        json.dump(prompt_lock, handle, indent=2)

    print("Selected prompt before held-out evaluation:", SELECTED_PROMPT)
    display(prompt_selection)

else:
    required_metrics = [
        "development_metrics_all_prompts_models.csv",
        "development_prompt_selection.csv",
        "selected_prompt_before_heldout.json",
    ]
    missing_metrics = [name for name in required_metrics if not (TABLE_DIR / name).is_file()]
    if missing_metrics:
        raise FileNotFoundError(
            "Saved-results mode is missing development outputs: "
            + ", ".join(missing_metrics)
        )
    development_metrics = pd.read_csv(
        TABLE_DIR / "development_metrics_all_prompts_models.csv"
    )
    prompt_selection = pd.read_csv(TABLE_DIR / "development_prompt_selection.csv")
    recomputed_selection = summarise_prompt_selection(development_metrics)
    pd.testing.assert_frame_equal(
        prompt_selection.reset_index(drop=True),
        recomputed_selection.reset_index(drop=True),
        check_dtype=False, atol=1e-12, rtol=1e-12,
    )
    with open(TABLE_DIR / "selected_prompt_before_heldout.json") as handle:
        prompt_lock = json.load(handle)
    SELECTED_PROMPT = prompt_lock["selected_prompt"]
    if SELECTED_PROMPT != EXPECTED_SELECTED_PROMPT:
        raise ValueError(
            f"Locked prompt is {SELECTED_PROMPT!r}; expected "
            f"{EXPECTED_SELECTED_PROMPT!r} from development selection."
        )
    if int(prompt_lock.get("sampling_seed", -1)) != SAMPLING_SEED:
        raise ValueError("Prompt lock does not use the fixed sampling seed.")
    selected_rows = prompt_selection[
        prompt_selection["selected"].astype(str).str.lower().eq("true")
    ]
    if len(selected_rows) != 1 or selected_rows.iloc[0]["prompt"] != SELECTED_PROMPT:
        raise ValueError("Prompt-selection table disagrees with the locked prompt JSON.")
    locked_thresholds = {
        key: float(value) for key, value in prompt_lock["encoder_thresholds"].items()
    }
    if any(not np.isclose(THRESHOLD[key], value) for key, value in locked_thresholds.items()):
        raise ValueError("Encoder thresholds disagree with the locked prompt JSON.")
    print("Selected prompt before held-out evaluation:", SELECTED_PROMPT)
    display(prompt_selection)


## Held-out LLM predictions

The prompt choice and encoder thresholds have already been written to disk before this cell runs.

In [ ]:
if RERUN_MODELS:
    run_llm_stage("heldout", heldout)
else:
    print("Held-out inference skipped; using the frozen result bundle.")


## Held-out encoder scores and final performance tables

In [ ]:
if RERUN_MODELS:
    heldout_encoder_scores = {
        "scibert_sim": scibert_scores(heldout, "heldout"),
        "sbert_minilm_sim": minilm_scores(heldout, "heldout"),
    }
    heldout_metrics = collect_metrics("heldout", heldout, heldout_encoder_scores)
    heldout_metrics.to_csv(TABLE_DIR / "heldout_metrics_all_prompts_models.csv", index=False)

    primary_heldout_metrics = heldout_metrics[
        heldout_metrics["prompt"].eq(SELECTED_PROMPT)
    ].copy()
    primary_heldout_metrics["model_display"] = primary_heldout_metrics["model"].map(MODEL_LABELS)
    primary_heldout_metrics.to_csv(
        TABLE_DIR / "heldout_metrics_selected_prompt_primary.csv", index=False
    )

    development_metrics["model_display"] = development_metrics["model"].map(MODEL_LABELS)
    heldout_metrics["model_display"] = heldout_metrics["model"].map(MODEL_LABELS)
    all_metrics = pd.concat([development_metrics, heldout_metrics], ignore_index=True)
    all_metrics.to_csv(TABLE_DIR / "all_development_and_heldout_metrics.csv", index=False)

    print("Primary held-out performance for pre-selected prompt:", SELECTED_PROMPT)
    display(primary_heldout_metrics[[
        "model_display", "n", "parse_rate", "accuracy",
        "precision", "recall", "f1", "tn", "fp", "fn", "tp",
    ]])

else:
    heldout_metrics_path = TABLE_DIR / "heldout_metrics_all_prompts_models.csv"
    primary_metrics_path = TABLE_DIR / "heldout_metrics_selected_prompt_primary.csv"
    if not heldout_metrics_path.is_file() or not primary_metrics_path.is_file():
        raise FileNotFoundError(
            "Saved-results mode requires both held-out metric tables."
        )
    heldout_metrics = pd.read_csv(heldout_metrics_path)
    primary_heldout_metrics = pd.read_csv(primary_metrics_path)
    if set(heldout_metrics["prompt"]) != set(PROMPT_BUILDERS):
        raise ValueError("Held-out metrics do not contain all six prompt strategies.")
    if set(primary_heldout_metrics["prompt"]) != {SELECTED_PROMPT}:
        raise ValueError("Primary held-out metrics do not use the locked prompt.")

    development_metrics["model_display"] = development_metrics["model"].map(MODEL_LABELS)
    heldout_metrics["model_display"] = heldout_metrics["model"].map(MODEL_LABELS)
    all_metrics = pd.concat([development_metrics, heldout_metrics], ignore_index=True)
    all_metrics.to_csv(TABLE_DIR / "all_development_and_heldout_metrics.csv", index=False)

    print("Primary held-out performance for pre-selected prompt:", SELECTED_PROMPT)
    display(primary_heldout_metrics[[
        "model_display", "n", "parse_rate", "accuracy",
        "precision", "recall", "f1", "tn", "fp", "fn", "tp",
    ]])


## Combined predictions and pairwise agreement

Pairwise agreement is calculated only on rows parsed by both models. The accompanying `n` tables make the denominator explicit.

In [ ]:
def combined_predictions(split_name, frame, prompt_name, encoder_scores):
    combined = frame[["id", "title", "abstract", "year", "label"]].copy()
    combined["True_label"] = combined["label"].astype(bool)
    for _, tag, _ in LLM_SPECS:
        prediction = read_llm_prediction(split_name, prompt_name, tag)
        combined[tag] = prediction["prediction"]
        combined[f"{tag}_parse_ok"] = prediction["parse_ok"]
    for tag, score_frame in encoder_scores.items():
        combined[tag] = score_frame["score"].ge(THRESHOLD[tag]).to_numpy()
        combined[f"{tag}_parse_ok"] = True
    return combined


def pairwise_agreement(frame):
    agreement = pd.DataFrame(index=MODEL_TAGS, columns=MODEL_TAGS, dtype=float)
    compared = pd.DataFrame(index=MODEL_TAGS, columns=MODEL_TAGS, dtype=int)
    for model_a in MODEL_TAGS:
        for model_b in MODEL_TAGS:
            valid = (
                frame[f"{model_a}_parse_ok"].astype(bool)
                & frame[f"{model_b}_parse_ok"].astype(bool)
            )
            compared.loc[model_a, model_b] = int(valid.sum())
            agreement.loc[model_a, model_b] = (
                frame.loc[valid, model_a].astype(bool)
                .eq(frame.loc[valid, model_b].astype(bool))
                .mean()
            )
    return agreement, compared



if RERUN_MODELS:
    pairwise_tables = {}
    for prompt_name in PROMPT_BUILDERS:
        combined = combined_predictions(
            "heldout", heldout, prompt_name, heldout_encoder_scores
        )
        combined.to_csv(TABLE_DIR / f"predictions_heldout_{prompt_name}.csv", index=False)
        combined.to_csv(TABLE_DIR / f"predictions_{prompt_name}.csv", index=False)
        agreement, compared = pairwise_agreement(combined)
        agreement_percent = agreement * 100
        agreement_percent.to_csv(
            TABLE_DIR / f"pairwise_agreement_percent_{prompt_name}.csv"
        )
        compared.to_csv(TABLE_DIR / f"pairwise_agreement_n_{prompt_name}.csv")
        pairwise_tables[prompt_name] = agreement

    display(pairwise_tables[SELECTED_PROMPT] * 100)

else:
    pairwise_tables = {}
    recomputed_metric_rows = []
    llm_tags = {tag for _, tag, _ in LLM_SPECS}
    for prompt_name in PROMPT_BUILDERS:
        prediction_path = TABLE_DIR / f"predictions_heldout_{prompt_name}.csv"
        if not prediction_path.is_file():
            raise FileNotFoundError(prediction_path)
        combined = pd.read_csv(prediction_path, low_memory=False)
        required_prediction_columns = {
            "id", "label",
            *MODEL_TAGS,
            *(f"{tag}_parse_ok" for tag in MODEL_TAGS),
        }
        missing = sorted(required_prediction_columns - set(combined.columns))
        if missing:
            raise ValueError(f"{prediction_path.name} is missing columns: {missing}")
        if combined["id"].astype(str).tolist() != heldout["id"].astype(str).tolist():
            raise ValueError(f"{prediction_path.name} does not match the held-out split.")
        if not combined["label"].astype(int).reset_index(drop=True).equals(
            heldout["label"].astype(int).reset_index(drop=True)
        ):
            raise ValueError(f"{prediction_path.name} labels disagree with the held-out split.")
        for model_tag in MODEL_TAGS:
            recomputed_metric_rows.append(metric_row(
                combined["label"], combined[model_tag],
                combined[f"{model_tag}_parse_ok"].astype(str).str.lower().eq("true"),
                split="heldout", prompt=prompt_name, model=model_tag,
                model_type="LLM" if model_tag in llm_tags else "encoder baseline",
            ))

        agreement, compared = pairwise_agreement(combined)
        saved_agreement = pd.read_csv(
            TABLE_DIR / f"pairwise_agreement_percent_{prompt_name}.csv", index_col=0
        ) / 100
        saved_compared = pd.read_csv(
            TABLE_DIR / f"pairwise_agreement_n_{prompt_name}.csv", index_col=0
        )
        pd.testing.assert_frame_equal(
            agreement, saved_agreement,
            check_dtype=False, check_names=False, atol=1e-12, rtol=1e-12,
        )
        pd.testing.assert_frame_equal(
            compared, saved_compared,
            check_dtype=False, check_names=False,
        )
        combined.to_csv(TABLE_DIR / f"predictions_{prompt_name}.csv", index=False)
        pairwise_tables[prompt_name] = agreement

    recomputed_heldout_metrics = pd.DataFrame(recomputed_metric_rows)
    metric_columns = [
        "split", "prompt", "model", "model_type", "n", "n_parsed",
        "parse_rate", "accuracy", "precision", "recall", "f1",
        "tn", "fp", "fn", "tp",
    ]
    sort_columns = ["prompt", "model"]
    pd.testing.assert_frame_equal(
        heldout_metrics[metric_columns].sort_values(sort_columns).reset_index(drop=True),
        recomputed_heldout_metrics[metric_columns].sort_values(sort_columns).reset_index(drop=True),
        check_dtype=False, atol=1e-12, rtol=1e-12,
    )
    recomputed_primary = recomputed_heldout_metrics[
        recomputed_heldout_metrics["prompt"].eq(SELECTED_PROMPT)
    ].copy()
    pd.testing.assert_frame_equal(
        primary_heldout_metrics[metric_columns].sort_values(sort_columns).reset_index(drop=True),
        recomputed_primary[metric_columns].sort_values(sort_columns).reset_index(drop=True),
        check_dtype=False, atol=1e-12, rtol=1e-12,
    )
    heldout_metrics = recomputed_heldout_metrics
    primary_heldout_metrics = recomputed_primary
    heldout_metrics["model_display"] = heldout_metrics["model"].map(MODEL_LABELS)
    primary_heldout_metrics["model_display"] = (
        primary_heldout_metrics["model"].map(MODEL_LABELS)
    )
    all_metrics = pd.concat([development_metrics, heldout_metrics], ignore_index=True)
    all_metrics.to_csv(TABLE_DIR / "all_development_and_heldout_metrics.csv", index=False)

    display(pairwise_tables[SELECTED_PROMPT] * 100)


## Pairwise-agreement figures

In [ ]:
agreement_cmap = sns.light_palette(LCDS_PALETTE[0], as_cmap=True)

for prompt_name, agreement in pairwise_tables.items():
    figure, axis = plt.subplots(figsize=(8.2, 7.1))
    sns.heatmap(
        agreement.rename(index=MODEL_LABELS, columns=MODEL_LABELS),
        vmin=0, vmax=1, cmap=agreement_cmap, annot=True, fmt=".2f",
        square=True, cbar_kws={"label": "Pairwise agreement"}, ax=axis,
    )
    axis.set_title(f"Held-out pairwise agreement: {PROMPT_LABELS[prompt_name]}")
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / f"pairwise_agreement_{prompt_name}.png", dpi=300, bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"pairwise_agreement_{prompt_name}.pdf", bbox_inches="tight")
    plt.close(figure)

figure, axes = plt.subplots(2, 3, figsize=(21, 13))
axes = axes.ravel()
panel_tags = ["a.", "b.", "c.", "d.", "e.", "f."]
for axis, panel_tag, (prompt_name, agreement) in zip(
    axes, panel_tags, pairwise_tables.items()
):
    labelled = agreement.rename(index=MODEL_LABELS, columns=MODEL_LABELS)
    sns.heatmap(
        labelled, vmin=0, vmax=1, cmap=agreement_cmap,
        annot=True, fmt=".2f", square=True, cbar=False, ax=axis,
        annot_kws={"fontsize": 7.5},
    )
    axis.set_title(PROMPT_LABELS[prompt_name], fontsize=12)
    axis.text(-0.18, 1.08, panel_tag, transform=axis.transAxes, fontsize=15, fontweight="bold")
    axis.tick_params(labelsize=8)
colour_axis = figure.add_axes([0.92, 0.20, 0.015, 0.62])
scalar = plt.cm.ScalarMappable(norm=plt.Normalize(0, 1), cmap=agreement_cmap)
figure.colorbar(scalar, cax=colour_axis, label="Pairwise agreement")
figure.suptitle("Pairwise agreement across prompt strategies on the held-out test set", fontsize=16)
figure.subplots_adjust(left=0.08, right=0.90, top=0.93, bottom=0.07, wspace=0.40, hspace=0.42)
figure.savefig(FIGURE_DIR / "pairwise_agreement_all_six_prompts.png", dpi=300, bbox_inches="tight")
figure.savefig(FIGURE_DIR / "pairwise_agreement_all_six_prompts.pdf", bbox_inches="tight")
plt.show()

## Held-out precision, recall and F1 figures

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(20, 6.5))
for axis, metric in zip(axes, ["precision", "recall", "f1"]):
    table = heldout_metrics.pivot(index="prompt", columns="model", values=metric)
    table = table.reindex(index=list(PROMPT_BUILDERS), columns=MODEL_TAGS)
    table = table.rename(index=PROMPT_LABELS, columns=MODEL_LABELS)
    sns.heatmap(
        table, vmin=0, vmax=1, cmap=agreement_cmap,
        annot=True, fmt=".2f", cbar=metric == "f1", ax=axis,
        cbar_kws={"label": "Score"},
    )
    axis.set_title(metric.capitalize())
    axis.set_xlabel("")
    axis.set_ylabel("")
    axis.tick_params(axis="x", rotation=45, labelsize=8)
    axis.tick_params(axis="y", labelsize=8)
figure.suptitle("Held-out performance across prompts and classifiers", fontsize=16)
figure.tight_layout()
figure.savefig(FIGURE_DIR / "heldout_precision_recall_f1_heatmaps.png", dpi=300, bbox_inches="tight")
figure.savefig(FIGURE_DIR / "heldout_precision_recall_f1_heatmaps.pdf", bbox_inches="tight")
plt.show()

plot_frame = primary_heldout_metrics.melt(
    id_vars=["model_display"], value_vars=["precision", "recall", "f1"],
    var_name="metric", value_name="score",
)
figure, axis = plt.subplots(figsize=(11, 5.8))
sns.barplot(
    data=plot_frame, x="model_display", y="score", hue="metric",
    palette=LCDS_PALETTE[:3], ax=axis,
)
axis.set_ylim(0, 1)
axis.set_xlabel("")
axis.set_ylabel("Held-out score")
axis.set_title(f"Primary held-out performance: {PROMPT_LABELS[SELECTED_PROMPT]}")
axis.tick_params(axis="x", rotation=25)
axis.legend(title="Metric")
figure.tight_layout()
figure.savefig(FIGURE_DIR / "heldout_selected_prompt_performance.png", dpi=300, bbox_inches="tight")
figure.savefig(FIGURE_DIR / "heldout_selected_prompt_performance.pdf", bbox_inches="tight")
plt.show()

## Confusion matrices for six prompts and six classifiers

Each panel reports counts and row-normalised percentages. Unparsed LLM responses are treated as negative, matching the conservative deployment rule, while parsing coverage is reported separately in the metric tables.

In [ ]:
confusion_rows = []
confusion_lookup = {}
for prompt_name in PROMPT_BUILDERS:
    combined = pd.read_csv(TABLE_DIR / f"predictions_heldout_{prompt_name}.csv")
    truth = combined["label"].astype(bool).to_numpy()
    for model_tag in MODEL_TAGS:
        effective = combined[model_tag].map(
            {True: True, False: False, "True": True, "False": False}
        ).fillna(False).astype(bool).to_numpy()
        matrix = confusion_matrix(truth, effective, labels=[False, True])
        confusion_lookup[(prompt_name, model_tag)] = matrix
        tn, fp, fn, tp = matrix.ravel()
        confusion_rows.append({
            "prompt": prompt_name, "model": model_tag,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        })
confusion_counts = pd.DataFrame(confusion_rows)
confusion_path = TABLE_DIR / "heldout_confusion_counts_all_prompts_models.csv"
if not RERUN_MODELS:
    if not confusion_path.is_file():
        raise FileNotFoundError(confusion_path)
    saved_confusion = pd.read_csv(confusion_path)
    pd.testing.assert_frame_equal(
        saved_confusion.sort_values(["prompt", "model"]).reset_index(drop=True),
        confusion_counts.sort_values(["prompt", "model"]).reset_index(drop=True),
        check_dtype=False,
    )
confusion_counts.to_csv(confusion_path, index=False)


def draw_confusion(axis, matrix, title, show_y=True):
    normalised = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    axis.imshow(normalised, cmap=agreement_cmap, vmin=0, vmax=1)
    for row in range(2):
        for column in range(2):
            axis.text(
                column, row,
                f"{matrix[row, column]:,}\n({normalised[row, column]:.1%})",
                ha="center", va="center", fontsize=7.5,
            )
    axis.set_xticks([0, 1], ["Pred. negative", "Pred. positive"], fontsize=7)
    axis.set_yticks([0, 1], ["True negative", "True positive"] if show_y else ["", ""], fontsize=7)
    axis.set_title(title, fontsize=9)


figure, axes = plt.subplots(6, 6, figsize=(24, 23))
for row, prompt_name in enumerate(PROMPT_BUILDERS):
    for column, model_tag in enumerate(MODEL_TAGS):
        draw_confusion(
            axes[row, column],
            confusion_lookup[(prompt_name, model_tag)],
            MODEL_LABELS[model_tag] if row == 0 else "",
            show_y=column == 0,
        )
        if column == 0:
            axes[row, column].set_ylabel(PROMPT_LABELS[prompt_name], fontsize=10, fontweight="bold")
figure.suptitle("Held-out confusion matrices: six prompts by six classifiers", fontsize=18)
figure.tight_layout(rect=[0, 0, 1, 0.98])
figure.savefig(FIGURE_DIR / "heldout_confusion_matrices_6_prompts_6_models.png", dpi=300, bbox_inches="tight")
figure.savefig(FIGURE_DIR / "heldout_confusion_matrices_6_prompts_6_models.pdf", bbox_inches="tight")
plt.show()

for prompt_name in PROMPT_BUILDERS:
    figure, axes = plt.subplots(2, 3, figsize=(13, 9))
    for axis, model_tag in zip(axes.ravel(), MODEL_TAGS):
        draw_confusion(
            axis, confusion_lookup[(prompt_name, model_tag)],
            MODEL_LABELS[model_tag], show_y=True,
        )
    figure.suptitle(f"Held-out confusion matrices: {PROMPT_LABELS[prompt_name]}", fontsize=15)
    figure.tight_layout(rect=[0, 0, 1, 0.96])
    figure.savefig(FIGURE_DIR / f"heldout_confusion_matrices_{prompt_name}.png", dpi=300, bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"heldout_confusion_matrices_{prompt_name}.pdf", bbox_inches="tight")
    plt.close(figure)

display(confusion_counts.head(12))

## Output inventory

In [ ]:
inventory = []
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        inventory.append({
            "relative_path": str(path.relative_to(OUT_DIR)),
            "size_bytes": path.stat().st_size,
        })
inventory = pd.DataFrame(inventory)
inventory.to_csv(OUT_DIR / "output_inventory.csv", index=False)
print("Selected prompt:", SELECTED_PROMPT)
print("Output files:", len(inventory))
print("Primary held-out metrics:", TABLE_DIR / "heldout_metrics_selected_prompt_primary.csv")
print("Combined agreement figure:", FIGURE_DIR / "pairwise_agreement_all_six_prompts.png")
print("Combined confusion figure:", FIGURE_DIR / "heldout_confusion_matrices_6_prompts_6_models.png")
display(inventory.tail(30))